In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np

from tqdm import tqdm
from shapely import Polygon, make_valid, geometry

In [3]:
ressource_survey_path = "../../../resources/surveys/edgt_lyon"
cleaned_survey_path = "../../../results/surveys/edgt_lyon"
locations_path = "../../../resources/locations"

output_path = "../../../results/surveys/edgt_lyon/trips.geoparquet"

In [4]:
if "papermill" in locals():
    survey_path = papermill.input["survey"]
    spatial_path = papermill.input["spatial"]

    output_path = papermill.output[0]

In [5]:
# Load spatial data

gdf_housing: gpd.GeoDataFrame = gpd.read_file("%s/lyon_all_potential_housing.gpkg" % locations_path, layer="housing")
gdf_work: gpd.GeoDataFrame = gpd.read_file("%s/lyon_all_potential_work.gpkg" % locations_path, layer="work")
gdf_education: gpd.GeoDataFrame = gpd.read_file("%s/lyon_all_potential_education.gpkg" % locations_path, layer="education")
gdf_secondary: gpd.GeoDataFrame = gpd.read_file("%s/lyon_all_potential_secondary.gpkg" % locations_path, layer="secondary")

gdf_zones: gpd.GeoDataFrame = gpd.read_file("%s/EDGT_AML2015_ZF_GT.TAB" % ressource_survey_path).to_crs("EPSG:2154")

In [6]:
gdf_housing


,location_id,weight,geometry
0,home_0,1.0,POINT (848193.38 6563109.52)
1,home_1,1.0,POINT (848203.98 6563089.67)
2,home_2,1.0,POINT (848214.41 6563076.71)
3,home_3,1.0,POINT (848221.98 6562965.48)
4,home_4,0.5,POINT (848245.17 6562942.05)
...,...,...,...
1202292,home_1202292,1.0,POINT (914706.049 6455313.756)
1202293,home_1202293,1.0,POINT (844485.142 6521425.826)
1202294,home_1202294,1.0,POINT (842684.713 6518921.219)
1202295,home_1202295,1.0,POINT (843266.058 6519578.269)


In [7]:
df_households = pd.read_parquet("%s/households.parquet" % cleaned_survey_path)
df_households

,edgt_household_id,zone_id,household_id,number_of_cars,number_of_motorbikes,number_of_bicycles
0,10100184,101001,0,0,0,2
1,101001115,101001,1,1,0,1
2,1010022,101002,2,1,0,2
3,1010024,101002,3,0,0,0
4,1010025,101002,4,0,0,0
...,...,...,...,...,...,...
6610,712451647,712451,16356,1,0,0
6611,712451653,712451,16357,1,0,1
6612,712451686,712451,16358,1,0,3
6613,712451742,712451,16359,1,2,0


In [8]:
# Merge df_households with gdf_zones to get the geometries
gdf_zones["zone_id"] = gdf_zones["ZF2015_Nouveau_codage"].astype(int)


In [9]:
df_persons: pd.DataFrame = pd.read_parquet("%s/persons.parquet" % cleaned_survey_path)
df_trips: pd.DataFrame = pd.read_parquet("%s/trips.parquet" % cleaned_survey_path)

In [10]:
df_trips


,household_id,person_id,trip_id,mode,mode_code,euclidean_distance,travel_time,departure_time,origin_cell,destination_cell,origin_activity_type,destination_activity_type,is_valid
0,0,0,0,pt,31,1110,900.0,38700.0,101001,102001,home,other,True
1,0,0,1,pt,31,1110,900.0,42300.0,102001,101001,other,home,True
2,0,1,2,pt,31,2590,1200.0,30000.0,101001,104001,home,work,True
3,0,1,3,pt,33,2750,1500.0,63900.0,104001,101001,work,home,True
4,0,2,4,pt,31,2390,600.0,26880.0,101001,212003,home,education,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
99580,16359,36561,99580,car,21,3610,600.0,61200.0,712451,711006,home,leisure,True
99581,16359,36561,99581,car,21,3610,600.0,61800.0,711006,712451,leisure,home,True
99582,16360,36562,99582,car,21,770,300.0,27000.0,712451,712001,home,other,True
99583,16360,36562,99583,car,21,770,480.0,27420.0,712001,712451,other,work,True


In [11]:
df_trips.rename({
    "origin_cell": "origin_zone_id",
    "destination_cell": "destination_zone_id",
    "origin_activity": "origin_activity_id",
}, axis=1, inplace=True)

df_trips

,household_id,person_id,trip_id,mode,mode_code,euclidean_distance,travel_time,departure_time,origin_zone_id,destination_zone_id,origin_activity_type,destination_activity_type,is_valid
0,0,0,0,pt,31,1110,900.0,38700.0,101001,102001,home,other,True
1,0,0,1,pt,31,1110,900.0,42300.0,102001,101001,other,home,True
2,0,1,2,pt,31,2590,1200.0,30000.0,101001,104001,home,work,True
3,0,1,3,pt,33,2750,1500.0,63900.0,104001,101001,work,home,True
4,0,2,4,pt,31,2390,600.0,26880.0,101001,212003,home,education,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
99580,16359,36561,99580,car,21,3610,600.0,61200.0,712451,711006,home,leisure,True
99581,16359,36561,99581,car,21,3610,600.0,61800.0,711006,712451,leisure,home,True
99582,16360,36562,99582,car,21,770,300.0,27000.0,712451,712001,home,other,True
99583,16360,36562,99583,car,21,770,480.0,27420.0,712001,712451,other,work,True


In [12]:

if False:
    df_trips = df_trips.loc[:100].copy()

# Create a cache to store geometry for each person_id, zone_id, and activity
geometry_cache = {}

def get_cached_geometry(person_id, zone_id, activity):
    return geometry_cache.get((person_id, zone_id, activity))

def set_cached_geometry(person_id, zone_id, activity, geometry):
    geometry_cache[(person_id, zone_id, activity)] = geometry

# Function to select a random point based on weight
def process_trip_group(trip_group: pd.DataFrame) -> pd.Series:
    
    zone_id = trip_group.name[0]
    activity = trip_group.name[1]

    result = pd.Series([None] * len(trip_group), name="geometry")

    group_zone = gdf_zones[(gdf_zones["zone_id"] == zone_id) & (gdf_zones["geometry"].geometry.type != 'Point')]
    if group_zone.empty:
        group_zone = gdf_zones[(gdf_zones["zone_id"] == zone_id) & (gdf_zones["geometry"].geometry.type == 'Point')]
        if group_zone.empty:
            return result
        else:
            return  pd.Series([group_zone["geometry"].iloc[0]] * len(trip_group), name="geometry")

    group_zone_geometry = group_zone["geometry"].iloc[0]

    if not group_zone_geometry.is_valid:
        group_zone_geometry = make_valid(group_zone_geometry)

    weights = None

    if activity == "home":
        gdf_filtered: gpd.GeoDataFrame = gdf_housing[gdf_housing.within(group_zone_geometry)]
        if gdf_filtered.empty:
            gdf_filtered = gdf_housing.iloc[[gdf_housing.distance(group_zone_geometry).idxmin()]]
        weights = "weight"
    if activity == "work":
        gdf_filtered: gpd.GeoDataFrame = gdf_work[gdf_work.within(group_zone_geometry)]
        weights = "employees"
        if gdf_filtered.empty:
            gdf_filtered = gdf_work.iloc[[gdf_work.distance(group_zone_geometry).idxmin()]]
    elif activity == "education":
        gdf_filtered: gpd.GeoDataFrame = gdf_education[gdf_education.within(group_zone_geometry)]
        if gdf_filtered.empty:
            gdf_filtered = gdf_education.iloc[[gdf_education.distance(group_zone_geometry).idxmin()]]
        weights = "weight"
    elif activity in ["leisure", "shop", "other"]:
        mask = gdf_secondary.within(group_zone_geometry)
        mask &= gdf_secondary["activity_type"] == activity
        gdf_filtered: gpd.GeoDataFrame = gdf_secondary[mask]
        if gdf_filtered.empty:
            gdf_filtered = gdf_secondary.iloc[[gdf_secondary.distance(group_zone_geometry).idxmin()]]

    if gdf_filtered.empty:
        weights=None
        gdf_filtered: gpd.GeoDataFrame = gdf_secondary[gdf_secondary.within(group_zone_geometry)]
        if gdf_filtered.empty:
            gdf_filtered = gdf_secondary.iloc[[gdf_secondary.distance(group_zone_geometry).idxmin()]]
    
    result = gdf_filtered.sample(n=len(trip_group), weights=weights, replace=True)["geometry"].reset_index(drop=True)
    result = result.rename("geometry")

    # Use and/or update cache
    for idx in range(len(trip_group)):
        trip = trip_group.iloc[idx]
        person_id = trip["person_id"]
        cached_geometry = get_cached_geometry(person_id, zone_id, activity)
        if cached_geometry is None:
            set_cached_geometry(person_id, zone_id, activity, result.iloc[idx])
        else:
            result.iloc[idx] = cached_geometry
            pass

    return result

# NEEEED to sort first !!!
tqdm.pandas(desc="Calculating origin_geometry")
df_trips.sort_values(["origin_zone_id", "origin_activity_type"], inplace=True)
origin_geometry = df_trips.groupby(["origin_zone_id", "origin_activity_type"]).progress_apply(process_trip_group, include_groups=False).reset_index(drop=True)
df_trips["origin_geometry"] = origin_geometry.values

tqdm.pandas(desc="Calculating destination_geometry")
df_trips.sort_values(["destination_zone_id", "destination_activity_type"], inplace=True)
destination_geometry = df_trips.groupby(["destination_zone_id", "destination_activity_type"]).progress_apply(process_trip_group, include_groups=False).reset_index(drop=True)
df_trips["destination_geometry"] = destination_geometry.values

df_trips

Calculating origin_geometry:   0%|          | 0/7342 [00:00<?, ?it/s]

Calculating destination_geometry: 100%|██████████| 7277/7277 [13:00<00:00,  9.33it/s] 


,household_id,person_id,trip_id,mode,mode_code,euclidean_distance,travel_time,departure_time,origin_zone_id,destination_zone_id,origin_activity_type,destination_activity_type,is_valid,origin_geometry,destination_geometry
80797,12383,27096,80797,walk,1,290,300.0,45000.0,101001,101001,leisure,education,True,POINT (841519.35 6517440.14),POINT (841607.6 6518297.5)
91,19,33,91,walk,1,696,720.0,31500.0,101002,101001,home,education,True,POINT (841192.08 6517598.23),POINT (841607.6 6518297.5)
93,19,33,93,walk,1,2609,2700.0,49500.0,101002,101001,home,education,True,POINT (841192.08 6517598.23),POINT (841607.6 6518297.5)
20098,3330,6305,20098,pt,33,5720,1800.0,30000.0,136003,101001,home,education,True,POINT (847077.06 6519828.67),POINT (841607.6 6518297.5)
33050,5443,10716,33050,pt,31,4240,1800.0,26100.0,214007,101001,home,education,True,POINT (840126.61 6516710.94),POINT (841505.6435842451 6517298.579645115)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1167,165,309,1167,walk,1,116,120.0,50400.0,999090,999090,leisure,work,True,None,None
97038,15810,35271,97038,car_passenger,22,0,300.0,47100.0,999090,999090,leisure,work,True,None,None
80721,12369,27057,80721,car,21,0,720.0,64800.0,999100,999100,work,home,True,None,None
57935,8530,18193,57935,walk,1,580,600.0,51600.0,999100,999100,home,work,True,None,None


In [13]:

df_trips = df_trips[(~df_trips["origin_geometry"].isna()) & (~df_trips["destination_geometry"].isna())]

In [14]:
df_trips.loc[:, "trip_geometry"] = df_trips.apply(lambda x: geometry.LineString([x["origin_geometry"], x["destination_geometry"]]), axis=1)
gdf_trips = gpd.GeoDataFrame(df_trips, geometry="trip_geometry", crs="EPSG:2154")
gdf_trips["origin_geometry"] = gpd.GeoSeries(gdf_trips["origin_geometry"]).to_wkt()
gdf_trips["destination_geometry"] = gpd.GeoSeries(gdf_trips["destination_geometry"]).to_wkt()

gdf_trips["computed_distance"] = gdf_trips["trip_geometry"].length
gdf_trips["distance_error"] = (gdf_trips["euclidean_distance"] - gdf_trips["computed_distance"]).abs()

C:\Users\lebescond\AppData\Local\Temp\ipykernel_16564\703357709.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_trips.loc[:, "trip_geometry"] = df_trips.apply(lambda x: geometry.LineString([x["origin_geometry"], x["destination_geometry"]]), axis=1)


In [ ]:
gdf_trips["computed_distance"] = gdf_trips["trip_geometry"].length
gdf_trips["distance_error"] = (gdf_trips["euclidean_distance"] - gdf_trips["computed_distance"]).abs()

NameError: name 'gdf_trips' is not defined

In [23]:
import plotly.express as px

fig = px.histogram(
    gdf_trips,
    x="distance_error",
    color="mode",
    nbins=100,
    opacity=0.5,
    title="Distance Error Histogram per Mode",
    labels={"distance_error": "Distance Error", "count": "Frequency"},
)
fig.update_layout(barmode="overlay")
fig.show()

In [46]:
import plotly.express as px
gdf_trips["distance_error_relative"] = gdf_trips["computed_distance"] / gdf_trips["euclidean_distance"]  # Ensure distance_error is positive
fig = px.histogram(
    gdf_trips,
    x="distance_error_relative",
    color="mode",
    nbins=200,
    opacity=0.5,
    title="Distance Error Histogram per Mode",
    labels={"distance_error_relative": "Distance Error in percent", "count": "Frequency"},
)
fig.update_layout(barmode="overlay")
fig.show()

In [47]:
gdf_trips.loc[gdf_trips["trip_id"] == 4191]
# gdf_trips.sort_values("distance_error", ascending=False, inplace=True)
# gdf_trips.loc[gdf_trips["mode"] == "car"]


,household_id,person_id,trip_id,mode,euclidean_distance,travel_time,departure_time,origin_zone_id,destination_zone_id,origin_activity_type,destination_activity_type,is_valid,origin_geometry,destination_geometry,trip_geometry,computed_distance,distance_error,distance_error_relative
4191,665,1256,4191,car,2080,3600.0,16200.0,108001,226004,home,work,True,POINT (842371.511949 6524454.140899),POINT (842412.28 6521932.06),"LINESTRING (842371.512 6524454.141, 842412.28 ...",2522.410374,-442.410374,1.212697


In [18]:
gdf_trips.to_parquet("%s/trips.geoparquet" % cleaned_survey_path)

In [19]:
gdf_trips.to_file("%s/trips.geojson" % cleaned_survey_path, driver="GeoJSON")